# 🔍 RAG Query Agent

**Purpose:** Natural language query interface for Ghana healthcare facilities using RAG (Retrieval-Augmented Generation)

**Architecture:**
1. **Retrieval**: Vector search over facility descriptions using embeddings
2. **Augmentation**: Retrieve relevant facility context
3. **Generation**: LLM generates natural language answers with citations

**Capabilities:**
- "Find hospitals in Accra with emergency care"
- "Which regions lack maternal care facilities?"
- "Show me facilities with high capacity but no doctors"
- "What are the top anomalies in Upper East Region?"

**Data Sources:**
- `facilities_silver`: Clean facility data
- `facilities_enriched`: LLM-enriched procedures/equipment
- `facilities_anomalies`: Data quality issues

---

In [0]:
%pip install openai

In [0]:
# Configuration
CATALOG = "virtue_foundation"
SCHEMA = "ghana"
FACILITIES_TABLE = "facilities_silver"
ENRICHED_TABLE = "facilities_enriched"
ANOMALIES_TABLE = "facilities_anomalies"

# RAG Configuration
EMBEDDING_MODEL = "databricks-bge-large-en"  # Databricks embedding endpoint
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"  # Databricks LLM endpoint
TOP_K_RESULTS = 5  # Number of facilities to retrieve

print(f"✅ Configuration set")
print(f"   Catalog: {CATALOG}.{SCHEMA}")
print(f"   Embedding Model: {EMBEDDING_MODEL}")
print(f"   LLM Model: {LLM_MODEL}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json
from openai import OpenAI
import numpy as np

print("✅ Libraries imported")

## Load Facility Data

In [0]:
# Load facilities with key attributes
df_facilities = spark.table(f"{CATALOG}.{SCHEMA}.{FACILITIES_TABLE}")

# Create searchable text from facility attributes
df_search = df_facilities.select(
    "unique_id",
    "name",
    "address_city",
    "address_stateOrRegion",
    "facilityTypeId",
    "organizationDescription",
    "specialties",
    "procedure",
    "equipment",
    "capability",
    "numberDoctors",
    "capacity",
    "phone_numbers",
    "email"
).withColumn(
    "search_text",
    F.concat_ws(
        " ",
        F.coalesce(F.col("name"), F.lit("")),
        F.coalesce(F.col("facilityTypeId"), F.lit("")),
        F.coalesce(F.col("address_city"), F.lit("")),
        F.coalesce(F.col("address_stateOrRegion"), F.lit("")),
        F.coalesce(F.col("organizationDescription"), F.lit("")),
        F.array_join(F.coalesce(F.col("specialties"), F.array()), " "),
        F.array_join(F.coalesce(F.col("procedure"), F.array()), " "),
        F.array_join(F.coalesce(F.col("equipment"), F.array()), " "),
        F.array_join(F.coalesce(F.col("capability"), F.array()), " ")
    )
)

facility_count = df_search.count()
print(f"✅ Loaded {facility_count} facilities")

# Show sample
print("\nSample search text:")
display(df_search.select("name", "address_stateOrRegion", "search_text").limit(3))

## Embedding & Vector Search

In [0]:
def get_embedding_client():
    """Initialize OpenAI client for Databricks embedding endpoint."""
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    token = ctx.apiToken().get()
    workspace_url = ctx.apiUrl().get()
    
    return OpenAI(
        api_key=token,
        base_url=f"{workspace_url}/serving-endpoints"
    )

def generate_embedding(text: str, client: OpenAI) -> list:
    """Generate embedding vector for text using Databricks endpoint."""
    try:
        response = client.embeddings.create(
            input=text,
            model=EMBEDDING_MODEL
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Embedding error: {e}")
        return None

def cosine_similarity(vec1: list, vec2: list) -> float:
    """Calculate cosine similarity between two vectors."""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print("✅ Embedding functions defined")

In [0]:
def semantic_search(query: str, top_k: int = TOP_K_RESULTS) -> list:
    """
    Perform semantic search over facilities using vector embeddings.
    
    Args:
        query: Natural language query
        top_k: Number of results to return
        
    Returns:
        List of (facility_dict, similarity_score) tuples
    """
    client = get_embedding_client()
    
    # Generate query embedding
    query_embedding = generate_embedding(query, client)
    if query_embedding is None:
        return []
    
    # Get all facilities (in production, this would use a vector database)
    facilities = df_search.toPandas()
    
    # Calculate similarities
    results = []
    for idx, row in facilities.iterrows():
        search_text = row["search_text"]
        if not search_text or search_text.strip() == "":
            continue
            
        # Generate embedding for facility
        facility_embedding = generate_embedding(search_text, client)
        if facility_embedding is None:
            continue
        
        # Calculate similarity
        similarity = cosine_similarity(query_embedding, facility_embedding)
        
        # Store result
        results.append({
            "facility": row.to_dict(),
            "similarity": float(similarity)
        })
    
    # Sort by similarity and return top-k
    results.sort(key=lambda x: x["similarity"], reverse=True)
    return results[:top_k]

print("✅ Semantic search function defined")

## RAG Query Pipeline

In [0]:
def rag_query(question: str, top_k: int = TOP_K_RESULTS) -> dict:
    """
    Answer a question about Ghana healthcare facilities using RAG.
    
    Args:
        question: Natural language question
        top_k: Number of facilities to retrieve
        
    Returns:
        dict with answer, sources, and retrieval info
    """
    import time
    
    # Step 1: Retrieve relevant facilities
    retrieval_start = time.time()
    search_results = semantic_search(question, top_k)
    retrieval_time = time.time() - retrieval_start
    
    if not search_results:
        return {
            "answer": "I couldn't find any relevant facilities to answer your question.",
            "sources": [],
            "retrieval_time": retrieval_time,
            "generation_time": 0,
            "num_sources": 0,
            "success": False
        }
    
    # Step 2: Format context from retrieved facilities
    context_parts = []
    sources = []
    
    for idx, result in enumerate(search_results, 1):
        facility = result["facility"]
        similarity = result["similarity"]
        
        # Helper to check if array field has values
        def has_values(field):
            return field is not None and len(field) > 0
        
        # Format facility info
        specialties = facility.get('specialties')
        procedures = facility.get('procedure')
        equipment = facility.get('equipment')
        capabilities = facility.get('capability')
        phones = facility.get('phone_numbers')
        
        context = f"""Facility {idx}:
Name: {facility.get('name', 'N/A')}
Type: {facility.get('facilityTypeId', 'N/A')}
Location: {facility.get('address_city', 'N/A')}, {facility.get('address_stateOrRegion', 'N/A')}
Doctors: {facility.get('numberDoctors', 'N/A')}
Capacity: {facility.get('capacity', 'N/A')} beds
Description: {facility.get('organizationDescription', 'N/A')}
Specialties: {', '.join(specialties) if has_values(specialties) else 'None'}
Procedures: {', '.join(procedures[:5]) if has_values(procedures) else 'None'}
Equipment: {', '.join(equipment[:5]) if has_values(equipment) else 'None'}
Capabilities: {', '.join(capabilities[:5]) if has_values(capabilities) else 'None'}
Contact: {', '.join(phones) if has_values(phones) else 'N/A'}
Relevance: {similarity:.3f}
"""
        context_parts.append(context)
        
        # Store source
        sources.append({
            "name": facility.get('name', 'Unknown'),
            "location": f"{facility.get('address_city', '')}, {facility.get('address_stateOrRegion', '')}",
            "type": facility.get('facilityTypeId', 'N/A'),
            "similarity": similarity
        })
    
    # Step 3: Generate answer using LLM
    generation_start = time.time()
    
    # Join context parts with separator (cannot use backslash in f-string)
    separator = "\n---\n"
    context_text = separator.join(context_parts)
    
    prompt = f"""You are a healthcare data analyst assistant for Ghana. Answer the user's question based ONLY on the provided facility data. Be specific, cite facility names, and provide actionable insights.

User Question: {question}

Relevant Facilities:
{context_text}

Instructions:
- Answer the question directly and concisely
- Cite specific facility names when relevant
- If asked for counts or statistics, provide exact numbers
- If the data doesn't fully answer the question, acknowledge limitations
- Format lists clearly with bullet points
- Include location information when helpful

Answer:"""
    
    try:
        client = get_embedding_client()
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0.2
        )
        answer = response.choices[0].message.content.strip()
        success = True
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
        success = False
    
    generation_time = time.time() - generation_start
    
    return {
        "answer": answer,
        "sources": sources,
        "retrieval_time": retrieval_time,
        "generation_time": generation_time,
        "num_sources": len(sources),
        "success": success
    }

print("✅ RAG query function defined")

In [0]:
def display_rag_result(result: dict, question: str = None):
    """Pretty print RAG query results."""
    print("="*80)
    if question:
        print(f"💬 Question: {question}")
        print("="*80)
    
    print(f"\n💡 Answer:\n{result['answer']}")
    
    print(f"\n\n📊 Query Stats:")
    print(f"   Sources retrieved: {result['num_sources']}")
    print(f"   Retrieval time: {result['retrieval_time']:.2f}s")
    print(f"   Generation time: {result['generation_time']:.2f}s")
    print(f"   Total time: {result['retrieval_time'] + result['generation_time']:.2f}s")
    print(f"   Success: {'✅' if result['success'] else '❌'}")
    
    if result['sources']:
        print(f"\n\n🏛️ Top Sources:")
        for idx, source in enumerate(result['sources'], 1):
            print(f"   {idx}. {source['name']}")
            print(f"      Location: {source['location']}")
            print(f"      Type: {source['type']}")
            print(f"      Relevance: {source['similarity']:.3f}")
            print()
    
    print("="*80)

print("✅ Display function defined")

## Example Queries

In [0]:
# Example 1: Find hospitals with specific capabilities
question1 = "Show me hospitals in Accra with emergency care capabilities"

print(f"🔍 Querying: {question1}\n")
result1 = rag_query(question1, top_k=3)
display_rag_result(result1, question1)

In [0]:
# Example 2: Regional analysis
question2 = "Which regions have the fewest healthcare facilities?"

print(f"🔍 Querying: {question2}\n")
result2 = rag_query(question2, top_k=5)
display_rag_result(result2, question2)

In [0]:
# Example 3: Specialized services
question3 = "Find facilities that provide maternal and pediatric care"

print(f"🔍 Querying: {question3}\n")
result3 = rag_query(question3, top_k=5)
display_rag_result(result3, question3)

In [0]:
# Example 4: Resource availability
question4 = "What facilities have high bed capacity but low doctor count?"

print(f"🔍 Querying: {question4}\n")
result4 = rag_query(question4, top_k=5)
display_rag_result(result4, question4)

## Interactive Query Interface

In [0]:
# Interactive query - modify the question below
custom_question = "Tell me about healthcare facilities in the Northern Region"

print(f"🔍 Custom Query: {custom_question}\n")
custom_result = rag_query(custom_question, top_k=5)
display_rag_result(custom_result, custom_question)

## Performance & Optimization Notes

**Current Implementation:**
- In-memory vector search over all facilities
- Suitable for ~1000 facilities
- Embeddings generated on-the-fly (slow)

**Production Optimizations:**

1. **Vector Database Integration:**
   - Pre-compute and store embeddings in Databricks Vector Search
   - Sub-second retrieval at scale
   - Automatic index updates

2. **Embedding Caching:**
   - Cache facility embeddings in Delta table
   - Regenerate only when facility data changes
   - 10-100x faster queries

3. **Hybrid Search:**
   - Combine vector search with keyword filters
   - Filter by region/type before semantic search
   - Improve relevance and speed

4. **Batch Processing:**
   - Generate embeddings in parallel batches
   - Use Spark UDFs for distributed processing
   - Handle millions of facilities

5. **Query Optimization:**
   - Cache common queries
   - Use smaller embedding models for lower latency
   - Implement query rewriting for better retrieval

**Next Steps:**
- Integrate with Databricks Vector Search Index
- Add metadata filtering (region, type, capacity ranges)
- Implement multi-turn conversations with memory
- Add query analytics and feedback loop